In [1]:
!pip install pbr importRosbag expelliarmus --no-deps -q
!pip install tonic --no-deps -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.9/131.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 2.7 MB/s eta 0:00:00


In [2]:
#pip install tonic

In [3]:
"""
=============================================================================
FULL ABLATION STUDY — PC-SNN with HH / LIF / Analog Neurons
=============================================================================
Covers Tables 5.2 – 5.6 of the thesis:
  Table 5.2  — MNIST      ablations  (inference mode, neuron model, encoding)
  Table 5.3  — FMNIST     ablations  (inference mode, neuron model, encoding)
  Table 5.4  — KMNIST     ablations  (inference mode, neuron model, encoding)
  Table 5.5  — N-MNIST    ablations  (representation, inference, neuron model)
  Table 5.6  — Caltech    ablations  (neuron model, encoding → AUC, energy, SPS, SR)

Every variant is implemented under a common ablation framework; N-MNIST uses a dedicated temporal event pipeline.
All models share identical hyperparameters except the single factor being ablated.

Usage:
    python ablation_study.py                        # run everything
    python ablation_study.py --datasets MNIST KMNIST # run subset
    python ablation_study.py --skip_nmnist          # skip neuromorphic (slow)

Outputs:
    ablation_results/
        table_5_2_MNIST.csv
        table_5_3_FMNIST.csv
        table_5_4_KMNIST.csv
        table_5_5_NMNIST.csv
        table_5_6_Caltech.csv
        ablation_summary.txt
=============================================================================
"""

import os
import sys
import time
import copy
import csv
import math
from dataclasses import dataclass, field, asdict
from typing import List, Tuple, Optional, Dict, Any, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Dataset, Subset
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────────────────
# 0.  UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

def set_seed(seed: int = 42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def default_device():
    return "cuda" if torch.cuda.is_available() else "cpu"

def one_hot(y: torch.Tensor, num_classes: int) -> torch.Tensor:
    return F.one_hot(y.long(), num_classes=num_classes).float()

os.makedirs("ablation_results", exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATASET LOADERS
# ─────────────────────────────────────────────────────────────────────────────

def _patch_kmnist_mirrors():
    """
    The official KMNIST mirror (codh.rois.ac.jp) is frequently down.
    Patch torchvision's KMNIST class to use the GitHub release mirror instead,
    which is stable and fast on Colab.
    """
    KMNIST_GITHUB_MIRRORS = [
        "https://github.com/rois-codh/kmnist/releases/download/v1.0/",
        "http://codh.rois.ac.jp/kmnist/dataset/kmnist/",   # original (kept as fallback)
    ]
    # Only patch if the first mirror is not already the GitHub one
    if datasets.KMNIST.mirrors[0] != KMNIST_GITHUB_MIRRORS[0]:
        datasets.KMNIST.mirrors = KMNIST_GITHUB_MIRRORS


def _download_kmnist_manual(root: str):
    """
    Last-resort manual download of KMNIST directly from GitHub releases
    using urllib, bypassing torchvision's download logic entirely.
    Only runs if the files are not already present.
    """
    import urllib.request
    import gzip
    import shutil

    kmnist_dir = os.path.join(root, "KMNIST", "raw")
    os.makedirs(kmnist_dir, exist_ok=True)

    files = {
        "train-images-idx3-ubyte.gz": "https://github.com/rois-codh/kmnist/releases/download/v1.0/train-images-idx3-ubyte.gz",
        "train-labels-idx1-ubyte.gz": "https://github.com/rois-codh/kmnist/releases/download/v1.0/train-labels-idx1-ubyte.gz",
        "t10k-images-idx3-ubyte.gz":  "https://github.com/rois-codh/kmnist/releases/download/v1.0/t10k-images-idx3-ubyte.gz",
        "t10k-labels-idx1-ubyte.gz":  "https://github.com/rois-codh/kmnist/releases/download/v1.0/t10k-labels-idx1-ubyte.gz",
    }

    for fname, url in files.items():
        out_gz   = os.path.join(kmnist_dir, fname)
        out_raw  = os.path.join(kmnist_dir, fname.replace(".gz", ""))

        if os.path.exists(out_raw):
            continue  # already extracted, skip

        if not os.path.exists(out_gz):
            print(f"  [KMNIST] Downloading {fname} ...")
            try:
                urllib.request.urlretrieve(url, out_gz)
            except Exception as e:
                raise RuntimeError(
                    f"KMNIST manual download failed for {fname}.\n"
                    f"URL tried: {url}\nError: {e}\n\n"
                    "Please download KMNIST manually from:\n"
                    "  https://github.com/rois-codh/kmnist/releases/tag/v1.0\n"
                    f"and place the extracted files in: {kmnist_dir}"
                )

        # extract .gz → raw binary
        with gzip.open(out_gz, "rb") as f_in, open(out_raw, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        os.remove(out_gz)

    print("  [KMNIST] Files ready.")


def _load_kmnist(root: str, tfm):
    """
    Try loading KMNIST with torchvision (patched mirrors first),
    then fall back to manual GitHub download if needed.
    """
    _patch_kmnist_mirrors()

    # Attempt 1: torchvision with patched mirrors
    try:
        train = datasets.KMNIST(root, train=True,  download=True, transform=tfm)
        test  = datasets.KMNIST(root, train=False, download=True, transform=tfm)
        return train, test
    except Exception as e1:
        print(f"  [KMNIST] torchvision download failed ({e1}). "
              f"Trying manual GitHub download ...")

    # Attempt 2: manual urllib download from GitHub releases
    try:
        _download_kmnist_manual(root)
        train = datasets.KMNIST(root, train=True,  download=False, transform=tfm)
        test  = datasets.KMNIST(root, train=False, download=False, transform=tfm)
        return train, test
    except Exception as e2:
        raise RuntimeError(
            f"Could not load KMNIST after all fallbacks.\n"
            f"Last error: {e2}\n\n"
            "Manual fix: download files from\n"
            "  https://github.com/rois-codh/kmnist/releases/tag/v1.0\n"
            f"and place extracted binaries in: {os.path.join(root, 'KMNIST', 'raw')}"
        )


def get_loaders(
    dataset_name: str,
    batch_size: int = 128,
    root: str = "./data",
    device: str = "cpu",
    val_ratio: float = 0.1,
    seed: int = 42,
):
    """Standard loader for MNIST / FMNIST / KMNIST."""
    tfm = transforms.Compose([transforms.ToTensor()])
    ds  = dataset_name.upper()

    if ds == "KMNIST":
        train_full, test_ds = _load_kmnist(root, tfm)
    elif ds in ("FMNIST", "FASHIONMNIST"):
        train_full = datasets.FashionMNIST(root, train=True,  download=True, transform=tfm)
        test_ds    = datasets.FashionMNIST(root, train=False, download=True, transform=tfm)
    else:  # MNIST
        train_full = datasets.MNIST(root, train=True,  download=True, transform=tfm)
        test_ds    = datasets.MNIST(root, train=False, download=True, transform=tfm)

    n_total  = len(train_full)
    n_val    = max(1, int(round(val_ratio * n_total)))
    n_train  = n_total - n_val
    g        = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(train_full, [n_train, n_val], generator=g)

    use_cuda = device.startswith("cuda") and torch.cuda.is_available()
    nw       = 2 if use_cuda else 0
    pin      = use_cuda
    kw       = dict(num_workers=nw, pin_memory=pin, persistent_workers=(nw > 0))

    return (
        DataLoader(train_ds, batch_size, shuffle=True,  **kw),
        DataLoader(val_ds,   batch_size, shuffle=False, **kw),
        DataLoader(test_ds,  batch_size, shuffle=False, **kw),
    )


# ── N-MNIST (neuromorphic) ────────────────────────────────────────────────────
# Canonical representation for N-MNIST:
#   real N-MNIST event stream -> 25 temporal bins -> polarity merge
#   -> [T, 1156] event frames.
# No MNIST fallback is used: a failed N-MNIST load must fail loudly.

class NMNISTTemporalDataset(Dataset):
    """
    Real N-MNIST represented as T temporal frames.

    Per sample:
        x: [T, 1156]
           where each 34x34 frame has positive/negative polarity merged.
        y: integer class label.
    """

    def __init__(self, root="./data", train=True, steps_spk=25):
        try:
            import tonic
            import tonic.transforms as tonic_tfm
        except ImportError as exc:
            raise ImportError(
                "N-MNIST requires tonic. Install it with: pip install tonic"
            ) from exc

        sensor_size = tonic.datasets.NMNIST.sensor_size  # (34, 34, 2)
        frame_transform = tonic_tfm.ToFrame(
            sensor_size=sensor_size,
            n_time_bins=steps_spk,
        )

        try:
            self.ds = tonic.datasets.NMNIST(
                save_to=root,
                train=train,
                transform=frame_transform,
            )
        except Exception as exc:
            raise RuntimeError(
                "Could not load real N-MNIST with tonic. "
                "Do not fall back to MNIST for thesis experiments. "
                "Check the dataset download/cache and tonic installation."
            ) from exc

        self.steps_spk = steps_spk

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample, label = self.ds[idx]

        # sample: [T, 2, 34, 34]
        x = torch.as_tensor(sample, dtype=torch.float32)

        if x.ndim != 4 or x.shape[1:] != (2, 34, 34):
            raise RuntimeError(
                f"Unexpected N-MNIST frame shape: {tuple(x.shape)}"
            )

        # Merge positive and negative polarity:
        # [T, 2, 34, 34] -> [T, 34, 34]
        x = x.sum(dim=1)

        # Flatten spatial dimensions:
        # [T, 34, 34] -> [T, 1156]
        x = x.reshape(x.shape[0], -1)

        # Binary event/intensity representation.
        x = x.clamp(0, 1)

        return x, int(label)


def get_nmnist_loaders(
    batch_size=128,
    root="./data",
    device="cpu",
    steps_spk=25,
    val_ratio=0.1,
    seed=42,
):
    """Return real temporal N-MNIST loaders with batches shaped [B, T, 1156]."""
    train_full = NMNISTTemporalDataset(
        root=root, train=True, steps_spk=steps_spk
    )
    test_ds = NMNISTTemporalDataset(
        root=root, train=False, steps_spk=steps_spk
    )

    input_dim = 1156

    n_total = len(train_full)
    n_val = max(1, int(round(val_ratio * n_total)))
    n_train = n_total - n_val

    g = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(
        train_full, [n_train, n_val], generator=g
    )

    def collate(batch):
        xs, ys = zip(*batch)
        return torch.stack(xs), torch.tensor(ys, dtype=torch.long)

    # Keep workers at zero for tonic event datasets unless you have explicitly
    # verified multiprocessing stability in your environment.
    return (
        DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            collate_fn=collate,
        ),
        DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            collate_fn=collate,
        ),
        DataLoader(
            test_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            collate_fn=collate,
        ),
        input_dim,
    )


def nmnist_event_preprocess(x, device):
    """Keep temporal N-MNIST as [B, T, 1156]."""
    x = x.to(device).float().clamp(0, 1)
    if x.ndim != 3:
        raise RuntimeError(
            f"Expected N-MNIST batch [B,T,D], got {tuple(x.shape)}"
        )
    return x


def nmnist_static_preprocess(x, device):
    """
    Collapse temporal N-MNIST to a static [B, 1156] representation.
    Used only for the explicit representation ablations.
    """
    x = x.to(device).float().clamp(0, 1)
    if x.ndim != 3:
        raise RuntimeError(
            f"Expected N-MNIST batch [B,T,D], got {tuple(x.shape)}"
        )
    return x.mean(dim=1).clamp(0, 1)


# ── Caltech Faces vs Motorbikes ───────────────────────────────────────────────
class CaltechBinaryDataset(Dataset):
    """
    Binary dataset: class 0 = Faces, class 1 = Motorbikes.
    Downloads from torchvision's Caltech101 if available; otherwise
    simulates with random data of matching dimensionality.
    """
    def __init__(self, root, train=True, seed=42):
        self.data, self.targets = self._load(root, train, seed)

    def _load(self, root, train, seed):
        try:
            tfm = transforms.Compose([
                transforms.Resize((64, 64)),
                transforms.Grayscale(),
                transforms.ToTensor(),
            ])
            full = datasets.Caltech101(root, download=True, transform=tfm)
            # Filter to Faces_easy (idx 9) and Motorbikes (idx 43)
            # class_to_idx mapping varies — find dynamically
            c2i = {c.lower(): i for i, c in enumerate(full.categories)}
            face_idx  = c2i.get("faces_easy", c2i.get("faces", 9))
            moto_idx  = c2i.get("motorbikes", 43)
            indices   = [i for i, t in enumerate(full.y) if t in (face_idx, moto_idx)]
            rng       = np.random.default_rng(seed)
            rng.shuffle(indices)
            split     = int(0.9 * len(indices)) if train else len(indices)
            subset    = indices[:split] if train else indices[split:]
            data_list, label_list = [], []
            for idx in subset:
                img, lbl = full[idx]
                binary_lbl = 0 if lbl == face_idx else 1
                data_list.append(img.flatten())
                label_list.append(binary_lbl)
            return data_list, label_list
        except Exception:
            # Fallback: random binary classification data
            rng = np.random.default_rng(seed)
            n   = 800 if train else 200
            data_list   = [torch.tensor(rng.random(64*64), dtype=torch.float32) for _ in range(n)]
            label_list  = [int(rng.integers(0, 2)) for _ in range(n)]
            return data_list, label_list

    def __len__(self):  return len(self.targets)
    def __getitem__(self, idx): return self.data[idx], self.targets[idx]


def get_caltech_loaders(batch_size=64, root="./data", device="cpu", seed=42):
    train_ds = CaltechBinaryDataset(root, train=True,  seed=seed)
    test_ds  = CaltechBinaryDataset(root, train=False, seed=seed)
    input_dim = train_ds[0][0].shape[0]

    n_total = len(train_ds)
    n_val   = max(1, int(round(0.15 * n_total)))
    n_train = n_total - n_val
    g = torch.Generator().manual_seed(seed)
    tr_ds, val_ds = random_split(train_ds, [n_train, n_val], generator=g)

    kw = dict(num_workers=0)
    return (
        DataLoader(tr_ds,   batch_size, shuffle=True,  **kw),
        DataLoader(val_ds,  batch_size, shuffle=False, **kw),
        DataLoader(test_ds, batch_size, shuffle=False, **kw),
        input_dim,
    )


# ─────────────────────────────────────────────────────────────────────────────
# 2.  NEURON MODELS
# ─────────────────────────────────────────────────────────────────────────────

class HHNeuron(nn.Module):
    class Gate:
        def __init__(self, B, N, device):
            self.alpha = torch.zeros(B, N, device=device)
            self.beta  = torch.zeros(B, N, device=device)
            self.state = torch.zeros(B, N, device=device)
        def update(self, dt):
            s = self.state + dt * (self.alpha*(1-self.state) - self.beta*self.state)
            return s.clamp(0.0, 1.0)
        def set_inf(self):
            self.state = self.alpha / (self.alpha + self.beta + 1e-8)

    def __init__(self, N, dt=0.03, device="cpu", thr=0.8, reset=0.0, tau_ref=2.0):
        super().__init__()
        self.N, self.dt, self.device = N, float(dt), device
        self.thr, self.reset = float(thr), float(reset)
        self.refr_steps = max(1, int(round(tau_ref / dt)))
        for name, val in [("ENa",115.),("EK",-12.),("Eleak",10.6),
                          ("gNa",120.),("gK",36.),("gLeak",0.3),("Cm",1.)]:
            self.register_buffer(name, torch.tensor(val))
        self.reset_states(1)

    def reset_states(self, B):
        dev = self.device
        self.B = B
        self.Vm = torch.zeros(B, self.N, device=dev)
        self.m  = HHNeuron.Gate(B, self.N, dev)
        self.n  = HHNeuron.Gate(B, self.N, dev)
        self.h  = HHNeuron.Gate(B, self.N, dev)
        self._update_gates(self.Vm)
        self.m.set_inf(); self.n.set_inf(); self.h.set_inf()
        self.refr = torch.zeros(B, self.N, device=dev)

    def _update_gates(self, V):
        V = V.clamp(-100., 100.)
        self.n.alpha = 0.01*(10-V)/(torch.exp((10-V)/10)-1+1e-8)
        self.n.beta  = 0.125*torch.exp(-V/80.)
        self.m.alpha = 0.1*(25-V)/(torch.exp((25-V)/10)-1+1e-8)
        self.m.beta  = 4.*torch.exp(-V/18.)
        self.h.alpha = 0.07*torch.exp(-V/20.)
        self.h.beta  = 1./(torch.exp((30-V)/10)+1.)

    def forward(self, I):
        if I.shape[0] != self.B:
            self.reset_states(I.shape[0])
        self._update_gates(self.Vm)
        m = self.m.update(self.dt); n = self.n.update(self.dt); h = self.h.update(self.dt)
        INa  = m**3 * self.gNa * h * (self.Vm - self.ENa)
        IK   = n**4 * self.gK       * (self.Vm - self.EK)
        IL   = self.gLeak            * (self.Vm - self.Eleak)
        dV   = (I - INa - IK - IL) / self.Cm
        Vn   = self.Vm + self.dt * dV
        Vn   = torch.tanh(Vn / 30.) * 30.
        can  = (self.refr <= 0)
        spk  = ((Vn >= self.thr) & can).float()
        self.Vm = torch.where(spk.bool(), torch.full_like(Vn, self.reset), Vn)
        self.m.state = m; self.n.state = n; self.h.state = h
        self.refr = torch.where(spk.bool(),
                                torch.full_like(self.refr, float(self.refr_steps)),
                                (self.refr - 1.).clamp(min=0.))
        return spk, self.Vm


class LIFNeuron(nn.Module):
    def __init__(self, N, dt=0.03, device="cpu", thr=0.8, reset=0.0, tau=0.3, tau_ref=2.0):
        super().__init__()
        self.N, self.dt, self.device = N, float(dt), device
        self.thr, self.reset, self.tau = float(thr), float(reset), float(tau)
        self.refr_steps = max(1, int(round(tau_ref / dt)))
        self.reset_states(1)

    def reset_states(self, B):
        self.B = B
        self.Vm   = torch.zeros(B, self.N, device=self.device)
        self.refr = torch.zeros(B, self.N, device=self.device)

    def forward(self, I):
        if I.shape[0] != self.B:
            self.reset_states(I.shape[0])
        can  = (self.refr <= 0)
        dV   = (-self.Vm + I) * (self.dt / max(self.tau, 1e-6))
        Vn   = self.Vm + dV
        spk  = ((Vn >= self.thr) & can).float()
        self.Vm = torch.where(spk.bool(), torch.full_like(Vn, self.reset), Vn)
        self.refr = torch.where(spk.bool(),
                                torch.full_like(self.refr, float(self.refr_steps)),
                                (self.refr - 1.).clamp(min=0.))
        return spk, self.Vm


# ─────────────────────────────────────────────────────────────────────────────
# 3.  PC ACTIVATION
# ─────────────────────────────────────────────────────────────────────────────

def make_pc_activation(name: str):
    name = name.lower()
    if name == "sigmoid":
        return torch.sigmoid, lambda z: torch.sigmoid(z)*(1-torch.sigmoid(z))
    if name == "tanh":
        return torch.tanh, lambda z: 1 - torch.tanh(z)**2
    # default: clipped relu
    def f(z):      return z.clamp(0., 1.)
    def fp(z):     return ((z > 0.) & (z < 1.)).float()
    return f, fp


# ─────────────────────────────────────────────────────────────────────────────
# 4.  GENERIC PC-SNN  (supports HH / LIF / Analog)
# ─────────────────────────────────────────────────────────────────────────────

class PCSNNet(nn.Module):
    """
    Unified PC-SNN supporting:
      neuron_type = "hh"     -> HH neurons
      neuron_type = "lif"    -> LIF neurons
      neuron_type = "analog"  -> non-spiking sigmoid proxy

    input_encoding =
      "poisson"       -> static image -> Poisson spikes
      "latency_first" -> static image -> latency-coded spikes
      "event_direct"  -> temporal N-MNIST frames fed directly at each step
    """

    def __init__(
        self,
        layer_sizes: List[int],
        neuron_type: str = "hh",
        dt: float = 0.03,
        device: str = "cpu",
        current_gain: float = 30.0,
        I_bias: float = 2.0,
        thr: float = 0.8,
        lif_tau: float = 0.3,
        pc_activation: str = "relu",
        lr: float = 2e-4,
        weight_decay: float = 1e-4,
        input_encoding: str = "poisson",
        poisson_scale: float = 1.0,
    ):
        super().__init__()
        assert len(layer_sizes) >= 2
        assert neuron_type.lower() in ("hh", "lif", "analog")
        assert input_encoding.lower() in (
            "poisson", "latency_first", "event_direct"
        )

        self.device = torch.device(device)
        self.sizes = layer_sizes
        self.neuron_type = neuron_type.lower()
        self.dt = float(dt)
        self.current_gain = float(current_gain)
        self.I_bias = float(I_bias)
        self.input_encoding = input_encoding.lower()
        self.poisson_scale = float(poisson_scale)
        self.f, self.fprime = make_pc_activation(pc_activation)
        self.L = len(layer_sizes) - 1
        self.S = self.L

        self.syn = nn.ModuleList([
            nn.Linear(layer_sizes[i], layer_sizes[i + 1], bias=True)
            for i in range(self.L)
        ])
        for lin in self.syn:
            nn.init.xavier_uniform_(lin.weight, gain=0.5)
            nn.init.zeros_(lin.bias)

        self.cells = nn.ModuleList()
        if self.neuron_type == "hh":
            for i in range(self.S):
                self.cells.append(
                    HHNeuron(
                        layer_sizes[i + 1],
                        dt=dt,
                        device=device,
                        thr=thr,
                    )
                )
        elif self.neuron_type == "lif":
            for i in range(self.S):
                self.cells.append(
                    LIFNeuron(
                        layer_sizes[i + 1],
                        dt=dt,
                        device=device,
                        thr=thr,
                        tau=lif_tau,
                    )
                )

        self.opt = torch.optim.Adam(
            self.syn.parameters(),
            lr=lr,
            weight_decay=weight_decay,
        )
        self._last_spike_sums: Optional[List[torch.Tensor]] = None

    @torch.no_grad()
    def _encode_latency(self, x0: torch.Tensor, steps: int) -> torch.Tensor:
        lat = steps * (1.0 - x0.clamp(0, 1))
        return lat.clamp(0.0, float(steps))

    @torch.no_grad()
    def _build_spike_train(
        self,
        latencies: torch.Tensor,
        steps: int,
    ) -> torch.Tensor:
        tgrid = torch.arange(
            1, steps + 1, device=self.device
        ).view(1, 1, -1)
        return (latencies.unsqueeze(-1) <= tgrid).float()

    @torch.no_grad()
    def _forward_static_proxies(
        self,
        x0: torch.Tensor,
        steps_spk: int,
    ) -> List[torch.Tensor]:
        B = x0.size(0)

        if self.neuron_type == "analog":
            proxies = [x0]
            r = x0
            for i in range(self.L):
                z = F.linear(
                    r,
                    self.syn[i].weight,
                    self.syn[i].bias,
                )
                r = torch.sigmoid(z)
                proxies.append(r.clamp(0, 1))
            self._last_spike_sums = None
            return proxies

        for cell in self.cells:
            cell.reset_states(B)

        if self.input_encoding == "latency_first":
            lat = self._encode_latency(x0, steps_spk)
            spk_train = self._build_spike_train(lat, steps_spk)
            fired = torch.zeros_like(x0, dtype=torch.bool)
        elif self.input_encoding == "poisson":
            p = (x0 * self.poisson_scale).clamp(0, 1)
            spk_train = (
                torch.rand(
                    B,
                    x0.size(1),
                    steps_spk,
                    device=self.device,
                ) < p.unsqueeze(-1)
            ).float()
            fired = None
        else:
            raise RuntimeError(
                "event_direct requires a temporal [B,T,D] input."
            )

        spike_sums = [
            torch.zeros(
                B,
                self.sizes[i + 1],
                device=self.device,
            )
            for i in range(self.S)
        ]

        for t in range(steps_spk):
            if self.input_encoding == "poisson":
                inp = spk_train[:, :, t]
            else:
                inp = (spk_train[:, :, t] * (~fired)).float()
                fired.logical_or_(inp.bool())

            r = inp
            for i in range(self.S):
                h = F.linear(
                    r,
                    self.syn[i].weight,
                    self.syn[i].bias,
                )
                I = h * self.current_gain + self.I_bias
                spk, _ = self.cells[i](I)
                spike_sums[i] += spk
                r = spk

        self._last_spike_sums = spike_sums
        return [x0] + [
            (ss / float(steps_spk)).clamp(0, 1)
            for ss in spike_sums
        ]

    @torch.no_grad()
    def forward_proxies(
        self,
        x_in: torch.Tensor,
        steps_spk: int,
    ) -> List[torch.Tensor]:
        if self.input_encoding == "event_direct":
            if self.neuron_type == "analog":
                # For the static analog baseline, use the temporal mean.
                if x_in.ndim != 3:
                    raise RuntimeError(
                        "Analog event_direct baseline expects [B,T,D]."
                    )
                x0 = x_in.to(self.device).float().clamp(0, 1).mean(dim=1)
                return self._forward_static_proxies(x0, 1)

            if x_in.ndim != 3:
                raise RuntimeError(
                    f"event_direct expects [B,T,D], got {tuple(x_in.shape)}"
                )

            x_event = x_in.to(self.device).float().clamp(0, 1)
            steps_use = min(x_event.size(1), steps_spk)
            if steps_use <= 0:
                raise RuntimeError("N-MNIST has zero temporal steps.")

            x0 = x_event[:, :steps_use, :].mean(dim=1).clamp(0, 1)
            B = x0.size(0)

            for cell in self.cells:
                cell.reset_states(B)

            spike_sums = [
                torch.zeros(
                    B,
                    self.sizes[i + 1],
                    device=self.device,
                )
                for i in range(self.S)
            ]

            for t in range(steps_use):
                r = x_event[:, t, :]

                for i in range(self.S):
                    h = F.linear(
                        r,
                        self.syn[i].weight,
                        self.syn[i].bias,
                    )
                    I = h * self.current_gain + self.I_bias
                    spk, _ = self.cells[i](I)
                    spike_sums[i] += spk
                    r = spk

            self._last_spike_sums = spike_sums
            return [x0] + [
                (ss / float(steps_use)).clamp(0, 1)
                for ss in spike_sums
            ]

        x0 = x_in.to(self.device).clamp(0, 1)
        return self._forward_static_proxies(x0, steps_spk)

    def last_spike_sums(self):
        return self._last_spike_sums

    def pc_infer(
        self,
        x_init,
        y_target=None,
        T_infer=50,
        eta_x=0.05,
        clamp_output=True,
    ):
        L = self.L
        x = [xi.clone().detach().to(self.device) for xi in x_init]
        x[0] = x[0].clamp(0, 1)

        if clamp_output and y_target is not None:
            x[L] = y_target.clone().detach().to(self.device).clamp(0, 1)

        z_cache = [None] * L
        for _ in range(T_infer):
            e = [None] * (L + 1)
            e[0] = torch.zeros_like(x[0])

            for l in range(1, L + 1):
                idx = l - 1
                z = F.linear(
                    x[l - 1],
                    self.syn[idx].weight,
                    self.syn[idx].bias,
                )
                z_cache[idx] = z
                e[l] = x[l] - self.f(z)

            for l in range(1, L):
                fb = (
                    e[l + 1] * self.fprime(z_cache[l])
                ) @ self.syn[l].weight
                x[l] = (
                    x[l] - eta_x * (e[l] - fb)
                ).clamp_(0, 1)

            if not clamp_output:
                x[L] = (
                    x[L] - eta_x * e[L]
                ).clamp_(0, 1)

        energy = 0.0
        with torch.no_grad():
            for l in range(1, L + 1):
                idx = l - 1
                z = F.linear(
                    x[l - 1],
                    self.syn[idx].weight,
                    self.syn[idx].bias,
                )
                el = x[l] - self.f(z)
                energy += 0.5 * (el ** 2).mean().item()

        return x, e, z_cache, energy

    def pc_learn(self, x, e, z_cache):
        B = x[0].shape[0]
        self.opt.zero_grad()

        for idx in range(self.L):
            local = e[idx + 1] * self.fprime(z_cache[idx])
            self.syn[idx].weight.grad = -(local.T @ x[idx]) / B
            self.syn[idx].bias.grad = -local.mean(0)

        torch.nn.utils.clip_grad_norm_(self.syn.parameters(), 1.0)
        self.opt.step()

    def train_step(self, x_in, y_target, steps_spk, T_infer, eta_x):
        proxies = self.forward_proxies(x_in, steps_spk)
        x, e, z_cache, energy = self.pc_infer(
            proxies,
            y_target,
            T_infer,
            eta_x,
            True,
        )
        self.pc_learn(x, e, z_cache)
        return energy, proxies


# ─────────────────────────────────────────────────────────────────────────────
# 5.  METRICS
# ─────────────────────────────────────────────────────────────────────────────

class MetricAccumulator:
    def __init__(self, C, device="cpu"):
        self.C = C; self.device = device; self.reset()

    def reset(self):
        self.tp = torch.zeros(self.C, dtype=torch.long)
        self.fp = torch.zeros(self.C, dtype=torch.long)
        self.fn = torch.zeros(self.C, dtype=torch.long)
        self.correct = self.total = 0

    @torch.no_grad()
    def update(self, pred, y):
        pred = pred.view(-1).long(); y = y.view(-1).long()
        self.total   += y.numel()
        self.correct += int((pred == y).sum())
        tp = torch.bincount(pred[pred == y], minlength=self.C)
        pc = torch.bincount(pred,            minlength=self.C)
        tc = torch.bincount(y,               minlength=self.C)
        self.tp += tp; self.fp += pc - tp; self.fn += tc - tp

    def compute(self, eps=1e-8):
        tp = self.tp.float(); fp = self.fp.float(); fn = self.fn.float()
        P  = (tp / (tp+fp+eps)).mean().item()
        R  = (tp / (tp+fn+eps)).mean().item()
        F1 = (2*tp / (2*tp+fp+fn+eps)).mean().item()
        return {"acc": self.correct/max(self.total,1),
                "precision": P, "recall": R, "f1": F1}


# ─────────────────────────────────────────────────────────────────────────────
# 6.  ROC-AUC (binary + macro OvR)
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def compute_roc_auc(
    model, loader, device, steps_spk, T_infer, eta_x, num_classes,
    eval_seed=1234, batch_preprocess_fn=None
):
    """Returns macro-OvR AUC."""
    try:
        from sklearn.metrics import roc_auc_score
    except ImportError:
        return float("nan")

    model.eval()
    all_scores, all_labels = [], []
    with torch.random.fork_rng():
        torch.manual_seed(eval_seed)
        prep = batch_preprocess_fn or default_batch_preprocess
        for x, y in loader:
            x = prep(x, device)
            proxies = model.forward_proxies(x, steps_spk)
            x_settle, _, _, _ = model.pc_infer(proxies, None, T_infer, eta_x, False)
            scores = x_settle[-1].cpu().numpy()
            all_scores.append(scores)
            all_labels.append(y.numpy())

    scores = np.concatenate(all_scores)
    labels = np.concatenate(all_labels)
    try:
        if num_classes == 2:
            auc = roc_auc_score(labels, scores[:, 1])
        else:
            auc = roc_auc_score(labels, scores, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    return auc


# ─────────────────────────────────────────────────────────────────────────────
# 7.  SPIKE RATE & EVALUATION
# ─────────────────────────────────────────────────────────────────────────────


def default_batch_preprocess(x, device):
    return x.to(device).view(x.size(0), -1)


@torch.no_grad()
def spike_rate_epoch(
    model,
    loader,
    device,
    steps_spk,
    eval_seed=1234,
    batch_preprocess_fn=None,
):
    model.eval()
    prep = batch_preprocess_fn or default_batch_preprocess
    S = getattr(model, "S", 0)

    if S <= 0 or model.neuron_type == "analog":
        return {
            "per_layer": [],
            "total": 0.0,
            "spikes_per_sample": 0.0,
        }

    total_spk = [0.0] * S
    total_den = [0.0] * S
    n_samples = 0

    with torch.random.fork_rng():
        torch.manual_seed(eval_seed)

        for x, _ in loader:
            x = prep(x, device)
            B = x.size(0)

            model.forward_proxies(x, steps_spk)
            ss = model.last_spike_sums()

            if ss is None:
                continue

            # The actual number of temporal simulation steps may be smaller
            # than steps_spk if a malformed/truncated batch is encountered.
            effective_steps = x.size(1) if x.ndim == 3 else steps_spk
            effective_steps = min(effective_steps, steps_spk)

            for li in range(S):
                total_spk[li] += float(ss[li].sum())
                total_den[li] += float(
                    B * ss[li].shape[1] * effective_steps
                )

            n_samples += B

    per_layer = [
        total_spk[li] / max(total_den[li], 1.0)
        for li in range(S)
    ]
    total = sum(total_spk) / max(sum(total_den), 1.0)
    sps = sum(total_spk) / max(n_samples, 1)

    return {
        "per_layer": per_layer,
        "total": total,
        "spikes_per_sample": sps,
    }


@torch.no_grad()
def eval_epoch(
    model,
    loader,
    device,
    steps_spk,
    T_infer,
    eta_x,
    eval_mode="pc",
    eval_seed=1234,
    batch_preprocess_fn=None,
):
    model.eval()
    prep = batch_preprocess_fn or default_batch_preprocess
    C = model.sizes[-1]

    ff_acc = MetricAccumulator(C)
    pc_acc = MetricAccumulator(C)
    total_e, total = 0.0, 0

    with torch.random.fork_rng():
        torch.manual_seed(eval_seed)

        for x, y in loader:
            x = prep(x, device)
            y = y.to(device)
            B = x.size(0)

            proxies = model.forward_proxies(x, steps_spk)

            ff_pred = proxies[-1].argmax(1)
            ff_acc.update(ff_pred.cpu(), y.cpu())

            if eval_mode == "pc":
                xs, _, _, _ = model.pc_infer(
                    proxies,
                    None,
                    T_infer,
                    eta_x,
                    False,
                )
                pc_pred = xs[-1].argmax(1)
            else:
                pc_pred = ff_pred

            pc_acc.update(pc_pred.cpu(), y.cpu())

            # Supervised reconstruction/energy diagnostic.
            y_oh = one_hot(y, C)
            _, _, _, e = model.pc_infer(
                proxies,
                y_oh,
                T_infer,
                eta_x,
                True,
            )
            total_e += e * B
            total += B

    ff_m = ff_acc.compute()
    pc_m = pc_acc.compute()

    return {
        "ff_acc": ff_m["acc"],
        "ff_f1": ff_m["f1"],
        "pc_acc": pc_m["acc"],
        "pc_f1": pc_m["f1"],
        "pc_precision": pc_m["precision"],
        "pc_recall": pc_m["recall"],
        "pc_energy": total_e / max(total, 1),
    }


# ─────────────────────────────────────────────────────────────────────────────
# 8.  TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────

class EarlyStopper:
    def __init__(self, patience=3, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best = None
        self.bad = 0
        self.best_epoch = 0

    def step(self, val, epoch):
        if self.best is None or val > self.best + self.min_delta:
            self.best = val
            self.bad = 0
            self.best_epoch = epoch
            return False, True

        self.bad += 1
        return (self.bad >= self.patience), False


def train(
    model,
    train_loader,
    val_loader,
    device,
    steps_spk,
    T_infer_train,
    T_infer_eval,
    eta_x,
    epochs,
    eval_mode,
    eval_seed,
    patience,
    ckpt_path,
    verbose=False,
    batch_preprocess_fn=None,
):
    prep = batch_preprocess_fn or default_batch_preprocess
    stopper = EarlyStopper(patience=patience)
    best_val_acc = 0.0

    for epoch in range(1, epochs + 1):
        model.train()

        for x, y in train_loader:
            x = prep(x, device)
            y = y.to(device)

            model.train_step(
                x,
                one_hot(y, model.sizes[-1]),
                steps_spk,
                T_infer_train,
                eta_x,
            )

        val_stats = eval_epoch(
            model,
            val_loader,
            device,
            steps_spk,
            T_infer_eval,
            eta_x,
            eval_mode,
            eval_seed,
            batch_preprocess_fn,
        )

        monitor = (
            val_stats["pc_acc"]
            if eval_mode == "pc"
            else val_stats["ff_acc"]
        )

        stop, improved = stopper.step(monitor, epoch)

        if improved:
            torch.save(model.state_dict(), ckpt_path)
            best_val_acc = monitor

        if verbose:
            print(
                f"  Epoch {epoch:02d} | val_acc={monitor:.4f}"
                f"{' *' if improved else ''}"
            )

        if stop:
            if verbose:
                print(f"  Early stop at epoch {epoch}")
            break

    if os.path.exists(ckpt_path):
        model.load_state_dict(
            torch.load(ckpt_path, map_location=device)
        )

    return best_val_acc


# ─────────────────────────────────────────────────────────────────────────────
# 9.  SINGLE VARIANT RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def run_variant(
    variant_name: str,
    layer_sizes: List[int],
    neuron_type: str,
    input_encoding: str,
    eval_mode: str,
    train_loader,
    val_loader,
    test_loader,
    device: str,
    steps_spk: int = 50,
    T_infer_train: int = 100,
    T_infer_eval: int = 50,
    eta_x: float = 0.05,
    lr: float = 2e-4,
    weight_decay: float = 1e-4,
    current_gain: float = 30.0,
    I_bias: float = 2.0,
    thr: float = 0.8,
    lif_tau: float = 0.3,
    epochs: int = 15,
    patience: int = 3,
    eval_seed: int = 1234,
    seed: int = 42,
    verbose: bool = False,
    num_classes: int = 10,
    compute_auc: bool = False,
    batch_preprocess_fn=None,
) -> Dict[str, Any]:
    set_seed(seed)

    safe_name = variant_name.replace(" ", "_").replace("/", "_")
    ckpt = f"ablation_results/ckpt_{safe_name}.pt"

    model = PCSNNet(
        layer_sizes=layer_sizes,
        neuron_type=neuron_type,
        device=device,
        current_gain=current_gain,
        I_bias=I_bias,
        thr=thr,
        lif_tau=lif_tau,
        lr=lr,
        weight_decay=weight_decay,
        input_encoding=input_encoding,
    ).to(device)

    t0 = time.time()

    train(
        model,
        train_loader,
        val_loader,
        device,
        steps_spk,
        T_infer_train,
        T_infer_eval,
        eta_x,
        epochs,
        eval_mode,
        eval_seed,
        patience,
        ckpt,
        verbose,
        batch_preprocess_fn,
    )

    elapsed = time.time() - t0

    test_stats = eval_epoch(
        model,
        test_loader,
        device,
        steps_spk,
        T_infer_eval,
        eta_x,
        eval_mode,
        eval_seed,
        batch_preprocess_fn,
    )

    spike_stats = spike_rate_epoch(
        model,
        test_loader,
        device,
        steps_spk,
        eval_seed,
        batch_preprocess_fn,
    )

    auc = float("nan")
    if compute_auc:
        auc = compute_roc_auc(
            model,
            test_loader,
            device,
            steps_spk,
            T_infer_eval,
            eta_x,
            num_classes,
            eval_seed,
            batch_preprocess_fn,
        )

    acc = (
        test_stats["pc_acc"]
        if eval_mode == "pc"
        else test_stats["ff_acc"]
    )
    f1 = (
        test_stats["pc_f1"]
        if eval_mode == "pc"
        else test_stats["ff_f1"]
    )

    result = {
        "variant": variant_name,
        "neuron": neuron_type.upper(),
        "encoding": input_encoding,
        "inference": eval_mode.upper(),
        "acc": round(acc * 100, 2),
        "f1": round(f1 * 100, 2),
        "precision": round(test_stats["pc_precision"] * 100, 2),
        "recall": round(test_stats["pc_recall"] * 100, 2),
        "pc_energy": round(test_stats["pc_energy"], 6),
        "spike_rate": round(spike_stats["total"], 6),
        "sps": round(spike_stats["spikes_per_sample"], 2),
        "auc": round(auc, 4) if not math.isnan(auc) else "N/A",
        "train_time_s": round(elapsed, 1),
    }

    if verbose:
        print(
            f"  → Acc={result['acc']}%  F1={result['f1']}%  "
            f"SR={result['spike_rate']:.5f}  "
            f"SPS={result['sps']:.1f}"
        )

    return result


# ─────────────────────────────────────────────────────────────────────────────
# 10.  ABLATION SUITES  (Tables 5.2 – 5.6)
# ─────────────────────────────────────────────────────────────────────────────

# Shared hyperparameters (kept identical across all variants for fair comparison)
BASE = dict(
    steps_spk     = 50,
    T_infer_train = 100,
    T_infer_eval  = 50,
    eta_x         = 0.05,
    lr            = 2e-4,
    weight_decay  = 1e-4,
    current_gain  = 30.0,
    I_bias        = 2.0,
    thr           = 0.8,
    lif_tau       = 0.3,
    epochs        = 15,
    patience      = 3,
    eval_seed     = 1234,
    seed          = 42,
)


def ablate_standard(dataset_name, train_loader, val_loader, test_loader,
                    layer_sizes, device, table_name, verbose=False):
    """
    Tables 5.2 / 5.3 / 5.4:
    Factor 1: inference mode  (PC vs FF)
    Factor 2: neuron model    (HH, LIF, Analog)
    Factor 3: encoding        (poisson, latency_first)
    """
    variants = [
        # (variant_label,       neuron,   encoding,        eval_mode)
        ("HH+PC",               "hh",     "poisson",       "pc"),
        ("LIF+PC",              "lif",    "poisson",       "pc"),
        ("Analog+PC",           "analog", "poisson",       "pc"),
        ("HH+PC (latency)",     "hh",     "latency_first", "pc"),
        ("LIF+PC (latency)",    "lif",    "latency_first", "pc"),
    ]

    rows = []
    for vname, neuron, enc, inf_mode in variants:
        print(f"  [{dataset_name}] {vname} ...")
        r = run_variant(
            variant_name   = f"{dataset_name}_{vname}",
            layer_sizes    = layer_sizes,
            neuron_type    = neuron,
            input_encoding = enc,
            eval_mode      = inf_mode,
            train_loader   = train_loader,
            val_loader     = val_loader,
            test_loader    = test_loader,
            device         = device,
            verbose        = verbose,
            num_classes    = layer_sizes[-1],
            compute_auc    = False,
            **BASE,
        )
        rows.append(r)

    _save_csv(rows, f"ablation_results/{table_name}.csv")
    _print_table(rows, table_name)
    return rows


def ablate_nmnist(
    train_loader,
    val_loader,
    test_loader,
    input_dim,
    device,
    verbose=False,
):
    """
    Table 5.5 — N-MNIST ablations.

    Canonical representation:
        real N-MNIST -> 25 temporal frames -> 1156-dim frame

    Variants:
      1. HH + predictive-coding inference on temporal events
      2. Same HH model, feed-forward evaluation
      3. LIF + predictive-coding inference on temporal events
      4. HH + PC on time-averaged static events with Poisson encoding
      5. HH + PC on time-averaged static events with latency encoding

    The first three are the primary temporal comparison.
    The final two quantify the effect of discarding temporal structure.
    """
    H = 512

    nmnist_base = {
        **BASE,
        "steps_spk": 25,
    }

    variants = [
        (
            "HH+PC (event-direct)",
            "hh",
            "event_direct",
            "pc",
            nmnist_event_preprocess,
        ),
        (
            "HH+PC (event-direct, FF)",
            "hh",
            "event_direct",
            "ff",
            nmnist_event_preprocess,
        ),
        (
            "LIF+PC (event-direct)",
            "lif",
            "event_direct",
            "pc",
            nmnist_event_preprocess,
        ),
        (
            "Analog+PC (temporal-mean)",
            "analog",
            "event_direct",
            "pc",
            nmnist_event_preprocess,
        ),
        (
            "HH+PC (static-poisson)",
            "hh",
            "poisson",
            "pc",
            nmnist_static_preprocess,
        ),
        (
            "HH+PC (static-latency)",
            "hh",
            "latency_first",
            "pc",
            nmnist_static_preprocess,
        ),
    ]

    rows = []

    for vname, neuron, enc, inf_mode, prep in variants:
        print(f"  [NMNIST] {vname} ...")

        r = run_variant(
            variant_name=f"NMNIST_{vname}",
            layer_sizes=[input_dim, H, 10],
            neuron_type=neuron,
            input_encoding=enc,
            eval_mode=inf_mode,
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader,
            device=device,
            verbose=verbose,
            num_classes=10,
            compute_auc=False,
            batch_preprocess_fn=prep,
            **nmnist_base,
        )
        rows.append(r)

    _save_csv(
        rows,
        "ablation_results/table_5_5_NMNIST.csv",
    )
    _print_table(
        rows,
        "Table 5.5 — N-MNIST",
    )
    return rows




# ─────────────────────────────────────────────────────────────────────────────
# 11.  I/O HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _save_csv(rows, path):
    if not rows: return
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader(); w.writerows(rows)
    print(f"  Saved → {path}")


def _print_table(rows, title):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")
    header = f"{'Variant':<30} {'Neuron':<8} {'Enc':<14} {'Inf':<5} {'Acc%':>7} {'F1%':>7}"
    print(header)
    print("-"*70)
    for r in rows:
        print(f"{r['variant']:<30} {r['neuron']:<8} {r['encoding']:<14} "
              f"{r['inference']:<5} {r['acc']:>7.2f} {r['f1']:>7.2f}")
    print()


def write_summary(all_results: Dict[str, List]):
    path = "ablation_results/ablation_summary.txt"
    with open(path, "w") as f:
        f.write("PC-SNN ABLATION STUDY — FULL SUMMARY\n")
        f.write("="*70 + "\n\n")
        for table_name, rows in all_results.items():
            f.write(f"{table_name}\n")
            f.write("-"*70 + "\n")
            if not rows: continue
            keys = ["variant","neuron","encoding","inference","acc","f1",
                    "pc_energy","spike_rate","sps","auc"]
            f.write("  ".join(f"{k:>14}" for k in keys) + "\n")
            for r in rows:
                f.write("  ".join(f"{str(r.get(k,''))[:14]:>14}" for k in keys) + "\n")
            f.write("\n")
    print(f"\nFull summary saved → {path}")


# ─────────────────────────────────────────────────────────────────────────────
# 12.  CONFIGURATION  (edit this section to control what runs)
# ─────────────────────────────────────────────────────────────────────────────
#
#  ╔══════════════════════════════════════════════════════════════════╗
#  ║  COLAB USERS: edit the CFG dict below, then run this cell.      ║
#  ║  No command-line arguments needed — argparse is NOT used here.  ║
#  ╚══════════════════════════════════════════════════════════════════╝

CFG = dict(
    # Which datasets to ablate. Remove any you don't want.
    datasets     = ["MNIST","FMNIST"],

    # Set True to skip slow neuromorphic / Caltech datasets
    skip_nmnist  = False,
    skip_caltech = False,

    # Model / training knobs
    batch_size   = 128,
    hidden       = 512,     # hidden layer width
    epochs       = 15,      # max epochs per variant
    steps_spk    = 50,      # static-dataset HH/LIF simulation timesteps
                             # N-MNIST uses 25 temporal bins explicitly

    # Misc
    verbose      = True,    # print per-epoch accuracy
    data_root    = "./data",
)


# ─────────────────────────────────────────────────────────────────────────────
# 13.  ENTRY POINT  — works from both `python script.py` and Colab `exec()`
# ─────────────────────────────────────────────────────────────────────────────

def run_ablations(cfg: dict = CFG):
    """
    Main entry point.  Call as:
        run_ablations()              # uses CFG defaults above
        run_ablations({"datasets": ["MNIST"], "epochs": 5})  # override
    """
    # merge overrides into defaults
    c = {**CFG, **cfg}

    device = default_device()
    set_seed(42)

    print(f"\n{'='*70}")
    print(f"  PC-SNN ABLATION STUDY")
    print(f"  Device   : {device}")
    print(f"  Datasets : {c['datasets']}")
    print(f"  Epochs   : {c['epochs']}  |  Steps : {c['steps_spk']}"
          f"  |  Hidden : {c['hidden']}")
    print(f"{'='*70}\n")

    # push epoch / step counts into BASE so all variants pick them up
    BASE["epochs"]    = c["epochs"]
    BASE["steps_spk"] = c["steps_spk"]

    torch.backends.cudnn.benchmark = True
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    ROOT = c["data_root"]
    BS   = c["batch_size"]
    H    = c["hidden"]
    VB   = c["verbose"]
    all_results = {}

    # ── MNIST ─────────────────────────────────────────────────────────────
    if "MNIST" in c["datasets"]:
        print("\n>>> MNIST (Table 5.2)")
        tr, va, te = get_loaders("MNIST", BS, ROOT, device)
        rows = ablate_standard("MNIST", tr, va, te,
                               [784, H, 10], device,
                               "table_5_2_MNIST", VB)
        all_results["Table 5.2 — MNIST"] = rows

    # ── FMNIST ────────────────────────────────────────────────────────────
    if "FMNIST" in c["datasets"]:
        print("\n>>> FashionMNIST (Table 5.3)")
        tr, va, te = get_loaders("FMNIST", BS, ROOT, device)
        rows = ablate_standard("FMNIST", tr, va, te,
                               [784, H, 10], device,
                               "table_5_3_FMNIST", VB)
        all_results["Table 5.3 — FMNIST"] = rows

    # ── N-MNIST ───────────────────────────────────────────────────────────
    if "NMNIST" in c["datasets"] and not c["skip_nmnist"]:
        print("\n>>> N-MNIST (Table 5.5)")

        tr, va, te, in_dim = get_nmnist_loaders(
            batch_size=BS,
            root=ROOT,
            device=device,
            steps_spk=25,
            seed=42,
        )

        rows = ablate_nmnist(
            tr, va, te, in_dim,
            device, VB)

        all_results["Table 5.5 — N-MNIST"] = rows

    # ── Caltech ───────────────────────────────────────────────────────────
    if "Caltech" in c["datasets"] and not c["skip_caltech"]:
        print("\n>>> Caltech Faces vs Motorbikes (Table 5.6)")
        tr, va, te, in_dim = get_caltech_loaders(BS, ROOT, device)
        rows = ablate_caltech(tr, va, te, in_dim, device, VB)
        all_results["Table 5.6 — Caltech"] = rows

    # ── Summary ───────────────────────────────────────────────────────────
    write_summary(all_results)
    print("\n✓ All ablations complete. Results saved to ablation_results/")
    return all_results


# ── Runs automatically whether called as a script or exec()'d in Colab ────────
run_ablations()


  PC-SNN ABLATION STUDY
  Device   : cuda
  Datasets : ['MNIST', 'FMNIST']
  Epochs   : 15  |  Steps : 50  |  Hidden : 512


>>> MNIST (Table 5.2)


100%|██████████| 9.91M/9.91M [00:00<00:00, 39.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 903kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 6.82MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.58MB/s]


  [MNIST] HH+PC ...


/usr/lib/python3.12/contextlib.py:137: UserWarning: CUDA reports that you have 2 available devices, and you have used fork_rng without explicitly specifying which devices are being used. For safety, we initialize *every* CUDA device by default, which can be quite slow if you have a lot of CUDAs. If you know that you are only making use of a few CUDA devices, set the environment variable CUDA_VISIBLE_DEVICES or the 'devices' keyword argument of fork_rng with the set of devices you are actually using. For example, if you are using CPU only, set device.upper()_VISIBLE_DEVICES= or devices=[]; if you are using device 0 only, set CUDA_VISIBLE_DEVICES=0 or devices=[0].  To initialize all devices and suppress this warning, set the 'devices' keyword argument to `range(torch.cuda.device_count())`.
  return next(self.gen)


  Epoch 01 | val_acc=0.9418 *
  Epoch 02 | val_acc=0.9572 *
  Epoch 03 | val_acc=0.9630 *
  Epoch 04 | val_acc=0.9655 *
  Epoch 05 | val_acc=0.9695 *
  Epoch 06 | val_acc=0.9700 *
  Epoch 07 | val_acc=0.9718 *
  Epoch 08 | val_acc=0.9737 *
  Epoch 09 | val_acc=0.9730
  Epoch 10 | val_acc=0.9745 *
  Epoch 11 | val_acc=0.9738
  Epoch 12 | val_acc=0.9752 *
  Epoch 13 | val_acc=0.9740
  Epoch 14 | val_acc=0.9725
  Epoch 15 | val_acc=0.9758 *
  → Acc=97.88%  F1=97.87%  SR=0.00834  SPS=217.8
  [MNIST] LIF+PC ...
  Epoch 01 | val_acc=0.9412 *
  Epoch 02 | val_acc=0.9563 *
  Epoch 03 | val_acc=0.9637 *
  Epoch 04 | val_acc=0.9672 *
  Epoch 05 | val_acc=0.9700 *
  Epoch 06 | val_acc=0.9710 *
  Epoch 07 | val_acc=0.9727 *
  Epoch 08 | val_acc=0.9722
  Epoch 09 | val_acc=0.9760 *
  Epoch 10 | val_acc=0.9740
  Epoch 11 | val_acc=0.9745
  Epoch 12 | val_acc=0.9755
  Early stop at epoch 12
  → Acc=97.74%  F1=97.73%  SR=0.00886  SPS=231.2
  [MNIST] Analog+PC ...
  Epoch 01 | val_acc=0.8910 *
  Epoch 

100%|██████████| 26.4M/26.4M [00:01<00:00, 14.5MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 245kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.03MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 9.50MB/s]

  [FMNIST] HH+PC ...


  Epoch 01 | val_acc=0.8498 *
  Epoch 02 | val_acc=0.8705 *
  Epoch 03 | val_acc=0.8712 *
  Epoch 04 | val_acc=0.8762 *
  Epoch 05 | val_acc=0.8782 *
  Epoch 06 | val_acc=0.8802 *
  Epoch 07 | val_acc=0.8818 *
  Epoch 08 | val_acc=0.8840 *
  Epoch 09 | val_acc=0.8843 *
  Epoch 10 | val_acc=0.8847 *
  Epoch 11 | val_acc=0.8885 *
  Epoch 12 | val_acc=0.8773
  Epoch 13 | val_acc=0.8790
  Epoch 14 | val_acc=0.8842
  Early stop at epoch 14
  → Acc=87.73%  F1=87.46%  SR=0.00789  SPS=206.0
  [FMNIST] LIF+PC ...
  Epoch 01 | val_acc=0.8515 *
  Epoch 02 | val_acc=0.8640 *
  Epoch 03 | val_acc=0.8755 *
  Epoch 04 | val_acc=0.8732
  Epoch 05 | val_acc=0.8817 *
  Epoch 06 | val_acc=0.8807
  Epoch 07 | val_acc=0.8847 *
  Epoch 08 | val_acc=0.8855 *
  Epoch 09 | val_acc=0.8807
  Epoch 10 | val_acc=0.8872 *
  Epoch 11 | val_acc=0.8842
  Epoch 12 | val_acc=0.8877 *
  Epoch 13 | val_acc=0.8757
  Epoch 14 | val_acc=0.8792
  Epoch 15 | val_acc=0.8847
  Early stop at epoch 15
  → Acc=87.76%  F1=87.39%  SR

{'Table 5.2 — MNIST': [{'variant': 'MNIST_HH+PC',
   'neuron': 'HH',
   'encoding': 'poisson',
   'inference': 'PC',
   'acc': 97.88,
   'f1': 97.87,
   'precision': 97.88,
   'recall': 97.87,
   'pc_energy': 7.4e-05,
   'spike_rate': 0.008344,
   'sps': 217.78,
   'auc': 'N/A',
   'train_time_s': 1156.9},
  {'variant': 'MNIST_LIF+PC',
   'neuron': 'LIF',
   'encoding': 'poisson',
   'inference': 'PC',
   'acc': 97.74,
   'f1': 97.73,
   'precision': 97.73,
   'recall': 97.73,
   'pc_energy': 0.000126,
   'spike_rate': 0.00886,
   'sps': 231.25,
   'auc': 'N/A',
   'train_time_s': 447.4},
  {'variant': 'MNIST_Analog+PC',
   'neuron': 'ANALOG',
   'encoding': 'poisson',
   'inference': 'PC',
   'acc': 89.61,
   'f1': 89.6,
   'precision': 90.37,
   'recall': 89.47,
   'pc_energy': 0.00443,
   'spike_rate': 0.0,
   'sps': 0.0,
   'auc': 'N/A',
   'train_time_s': 83.4},
  {'variant': 'MNIST_HH+PC (latency)',
   'neuron': 'HH',
   'encoding': 'latency_first',
   'inference': 'PC',
   'acc'